<a href="https://colab.research.google.com/github/ektam9931/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ektam9931/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
Rule: Rank content higher when it shows stronger evidence of needing an update: older content receives a higher staleness score, and content with weaker CTR relative to its search position receives additional priority. The score is used for decision-support, not as a prediction of future performance.

Reason codes:
- STALE_CONTENT — content has been unchanged for a long time.
- LOW_CTR_FOR_POSITION — observed CTR is relatively weak for the content's search-position tier.
- STALE_AND_LOW_CTR — both signals are present.
- NO_CLEAR_SIGNAL — neither signal is strong enough to prioritize.
*Write the rule in plain words first. Then the reason codes it can output.*

In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [30]:
df.shape

(30000, 44)

In [31]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [32]:
df[["days_since_last_update", "ctr", "avg_position"]].describe()

,days_since_last_update,ctr,avg_position
count,30000.000000,30000.000000,30000.00000
mean,46.098300,0.510733,16.34238
std,42.078709,3.279162,15.21679
min,1.000000,0.000000,0.00000
25%,20.000000,0.000000,6.20000
50%,20.000000,0.070000,10.80000
75%,104.000000,0.290000,22.30000
max,373.000000,100.000000,245.00000


In [33]:
df.groupby("position_tier")["ctr"].agg(["count", "mean", "median"])

,count,mean,median
position_tier,,,
deep,1319,0.150212,0.00
page_1,11814,0.652467,0.16
page_3_5,7242,0.222484,0.03
striking,7304,0.323239,0.11
top_3,2321,1.483611,0.00


In [34]:


stale = df["days_since_last_update"] > 104


position_ctr_thresholds = {
    "top_3": 0.15,
    "page_1": 0.16,
    "striking": 0.13,
    "page_3_5": 0.08,
    "deep": 0.0975
}

low_ctr_for_position = (
    (df["ctr"] > 0) &
    df.apply(
        lambda row: row["ctr"] < position_ctr_thresholds.get(
            row["position_tier"], np.inf
        ),
        axis=1
    )
)

print("Stale:", stale.sum())
print("Low CTR for position:", low_ctr_for_position.sum())

Stale: 318
Low CTR for position: 3924


In [35]:
print("Both signals:", (stale & low_ctr_for_position).sum())
print("Stale only:", (stale & ~low_ctr_for_position).sum())
print("Low CTR only:", (~stale & low_ctr_for_position).sum())
print("Neither:", (~stale & ~low_ctr_for_position).sum())

Both signals: 19
Stale only: 299
Low CTR only: 3905
Neither: 25777


In [36]:
df.groupby("position_tier")["ctr"].quantile(0.25)

,ctr
position_tier,
deep,0.0
page_1,0.0
page_3_5,0.0
striking,0.0
top_3,0.0


In [37]:
df.groupby("position_tier")["ctr"].apply(lambda x: (x == 0).sum())

,ctr
position_tier,
deep,1107
page_1,4038
page_3_5,3425
striking,2860
top_3,1782


In [38]:
df[df["ctr"] > 0].groupby("position_tier")["ctr"].describe()

,count,mean,std,min,25%,50%,75%,max
position_tier,,,,,,,,
deep,212.0,0.934575,3.938375,0.01,0.0975,0.18,0.5125,50.00
page_1,7776.0,0.991286,3.763101,0.01,0.1600,0.31,0.6000,100.00
page_3_5,3817.0,0.422119,2.944557,0.01,0.0800,0.16,0.3100,100.00
striking,4444.0,0.531265,1.747171,0.01,0.1300,0.25,0.4800,33.33
top_3,539.0,6.388609,16.051681,0.01,0.1500,0.41,1.0850,100.00


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [39]:

stale = df["days_since_last_update"] > 104

position_ctr_thresholds = {
    "top_3": 0.15,
    "page_1": 0.16,
    "striking": 0.13,
    "page_3_5": 0.08,
    "deep": 0.0975
}

low_ctr_for_position = (
    (df["ctr"] > 0) &
    df.apply(
        lambda row: row["ctr"] < position_ctr_thresholds.get(
            row["position_tier"], np.inf
        ),
        axis=1
    )
)


df["score"] = (
    stale.astype(int) * 2
    + low_ctr_for_position.astype(int) * 1
)


df["reason_code"] = np.select(
    [
        stale & low_ctr_for_position,
        stale,
        low_ctr_for_position
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE_CONTENT",
        "LOW_CTR_FOR_POSITION"
    ],
    default="NO_CLEAR_SIGNAL"
)


df["action"] = np.where(
    df["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)


df = df.sort_values(
    ["score", "days_since_last_update"],
    ascending=[False, False]
).reset_index(drop=True)


print(df[["score", "reason_code", "action"]].head(10))


output_columns = [
    "content_id",
    "score",
    "reason_code",
    "action"
]

df[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved: work/outputs/baseline_action_score.csv")

   score        reason_code  action
0      3  STALE_AND_LOW_CTR  REVIEW
1      3  STALE_AND_LOW_CTR  REVIEW
2      3  STALE_AND_LOW_CTR  REVIEW
3      3  STALE_AND_LOW_CTR  REVIEW
4      3  STALE_AND_LOW_CTR  REVIEW
5      3  STALE_AND_LOW_CTR  REVIEW
6      3  STALE_AND_LOW_CTR  REVIEW
7      3  STALE_AND_LOW_CTR  REVIEW
8      3  STALE_AND_LOW_CTR  REVIEW
9      3  STALE_AND_LOW_CTR  REVIEW
Saved: work/outputs/baseline_action_score.csv


In [40]:
import os

os.makedirs("work/outputs", exist_ok=True)

df[["content_id", "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved: work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

| Rank | Content ID | Action | Reason code | Confidence note | What would make it wrong |
|---:|---|---|---|---|---|
| 1 | content_5feee3994adb3 | REVIEW | STALE_AND_LOW_CTR | High: 194 days stale and CTR is 0.01 at position 39.01. | The page may have a deliberate low-traffic role or other context not captured by these signals. |
| 2 | content_928af3e22c803 | REVIEW | STALE_AND_LOW_CTR | High: 193 days stale with CTR 0.12 at position 15.82. | The observed CTR may be acceptable for this specific query or content intent. |
| 3 | content_f1cdef3150473 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 on page 1. | The page may already satisfy its intended search intent despite the weak signal. |
| 4 | content_a8680d500da83 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.06 on page 1. | Position and CTR alone may not capture the page's actual business value. |
| 5 | content_cbffee46b6293 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.03 at position 42.25. | Deep ranking may explain the low CTR rather than content staleness. |
| 6 | content_e91ce7f0a0b33 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.10 at position 19.16. | The low CTR may be normal for the page's search intent. |
| 7 | content_9b28cf5ae4f33 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.06 despite a top-3 position. | The CTR threshold may not reflect the actual query or SERP context. |
| 8 | content_c4b8d24dcb3f | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 at position 14.48. | Other SERP factors could explain the observed CTR. |
| 9 | content_bef9eb4949af3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 on page 1. | The page may not need updating even though the two signals are present. |
| 10 | content_df1e8cee858c3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.09 on page 1. | The measured CTR may be acceptable for the page's specific intent. |
| 11 | content_323d0bdf70773 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.05 on page 1. | The low CTR may be caused by factors outside the content itself. |
| 12 | content_c3970ef960d53 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 at position 23.81. | The position itself may explain much of the low CTR. |
| 13 | content_14d64f61e5d93 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 at position 15.90. | The page may still be performing appropriately for its intended query. |
| 14 | content_a5dbb404bdc23 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.07 at position 18.71. | Low CTR may reflect search-result competition rather than stale content. |
| 15 | content_a05bb99e3fdf3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.08 at position 16.50. | The observed signal may not indicate that an update would improve performance. |
| 16 | content_97a36e9e956f3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.08 at position 15.10. | The CTR may be reasonable given the page's search context. |
| 17 | content_b08562686d223 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.14 at position 2.01. | A low CTR threshold may be inappropriate for this particular query or SERP. |
| 18 | content_6ac3ab740bbf3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.14 at position 14.60. | The page may have a valid reason for its observed CTR. |
| 19 | content_452a4e18212c3 | REVIEW | STALE_AND_LOW_CTR | High: 106 days stale with CTR 0.04 at position 24.21. | Its low position may be the main reason for the low CTR. |
| 20 | content_3f3576c295f52 | REVIEW | STALE_CONTENT | High: 373 days since update, but CTR is 100% at position 31.00. | The page may be intentionally unchanged and still performing well; staleness alone does not prove an update is needed. |

In [41]:
top20 = df.head(20)

top20[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "ctr",
        "position_tier",
        "avg_position"
    ]
]

,content_id,score,reason_code,action,days_since_last_update,ctr,position_tier,avg_position
0,content_5feee3994adb,3,STALE_AND_LOW_CTR,REVIEW,194,0.01,page_3_5,39.0
1,content_928af3e22c80,3,STALE_AND_LOW_CTR,REVIEW,193,0.12,striking,15.8
2,content_323d0bdf7077,3,STALE_AND_LOW_CTR,REVIEW,106,0.05,page_1,4.6
3,content_f1cdef315047,3,STALE_AND_LOW_CTR,REVIEW,106,0.07,page_1,6.2
4,content_c3970ef960d5,3,STALE_AND_LOW_CTR,REVIEW,106,0.07,page_3_5,23.8
5,content_9b28cf5ae4f3,3,STALE_AND_LOW_CTR,REVIEW,106,0.06,top_3,1.1
6,content_a8680d500da8,3,STALE_AND_LOW_CTR,REVIEW,106,0.06,page_1,9.7
7,content_c4b8d24dcb3f,3,STALE_AND_LOW_CTR,REVIEW,106,0.07,striking,14.4
8,content_14d64f61e5d9,3,STALE_AND_LOW_CTR,REVIEW,106,0.07,striking,15.9
9,content_a5dbb404bdc2,3,STALE_AND_LOW_CTR,REVIEW,106,0.07,page_1,8.7


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*### Weak Picks + Leakage Check

The weakest pick is the stale-only result at rank 20 (`content_3f3576c295f52`). It receives a review action because it has been unchanged for 373 days, but its observed CTR is 100%, so staleness alone does not establish that an update is needed. This is a useful example of where the baseline rule can be wrong.

The other top-20 rows are mostly supported by both observed signals: staleness and low non-zero CTR relative to position. These are decision-support signals rather than predictions of future performance.

### Leakage check

- The rule uses observed fields from the available dataset: `days_since_last_update`, `ctr`, and `position_tier`.
- No future-window fields were used to create the score.
- No product/client flags or external URLs were used.
- No label-derived outcome was used.
- The score is a baseline ranking rule, not a prediction of future performance.

In [42]:

scoring_inputs = [
    "days_since_last_update",
    "ctr",
    "position_tier"
]

print("Scoring inputs:")
print(scoring_inputs)

print("\nFuture-window fields used:")
future_window_fields = [
    c for c in scoring_inputs
    if any(x in c.lower() for x in ["future", "next", "post", "label"])
]
print(future_window_fields)

print("\nOutput shape:", df.shape)
print("Top-20 reviewed:", len(df.head(20)))

Scoring inputs:
['days_since_last_update', 'ctr', 'position_tier']

Future-window fields used:
[]

Output shape: (30000, 47)
Top-20 reviewed: 20


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.